In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/rtnhiuthanhthao/envi-model-final1/kaggle/working/envi-model-finalversion1/config.json
/kaggle/input/datasets/rtnhiuthanhthao/envi-model-final1/kaggle/working/envi-model-finalversion1/vocab.json
/kaggle/input/datasets/rtnhiuthanhthao/envi-model-final1/kaggle/working/envi-model-finalversion1/tokenizer_config.json
/kaggle/input/datasets/rtnhiuthanhthao/envi-model-final1/kaggle/working/envi-model-finalversion1/source.spm
/kaggle/input/datasets/rtnhiuthanhthao/envi-model-final1/kaggle/working/envi-model-finalversion1/model.safetensors
/kaggle/input/datasets/rtnhiuthanhthao/envi-model-final1/kaggle/working/envi-model-finalversion1/target.spm
/kaggle/input/datasets/rtnhiuthanhthao/envi-model-final1/kaggle/working/envi-model-finalversion1/generation_config.json


In [2]:
!pip install -q transformers datasets sentencepiece sacrebleu evaluate bert_score nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.6 MB/s eta 0:00:00


In [3]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,      # ← đã thêm lại
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed
)
import evaluate
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

set_seed(42)

In [4]:
from datasets import load_dataset

dataset       = load_dataset("ura-hcmut/PhoMT")
train_dataset = dataset["train"].select(range(140000))
valid_dataset = dataset["validation"].select(range(1268))

print(f"Train size : {len(train_dataset)}")
print(f"Valid size : {len(valid_dataset)}")
print(f"Sample     : {train_dataset[0]}")

README.md:   0%|          | 0.00/653 [00:00<?, ?B/s]

PhoMT_training.csv:   0%|          | 0.00/525M [00:00<?, ?B/s]

PhoMT_validation.csv: 0.00B [00:00, ?B/s]

PhoMT_test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2977999 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/18720 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/19151 [00:00<?, ? examples/s]

Train size : 140000
Valid size : 1268
Sample     : {'en': 'It begins with a countdown.', 'vi': 'Câu chuyện bắt đầu với buổi lễ đếm ngược.'}


In [5]:
model_name = "Helsinki-NLP/opus-mt-en-vi"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model     = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print(f"Model loaded : {model_name}")
print(f"Vocab size   : {tokenizer.vocab_size}")
print(f"Model params : {model.num_parameters():,}")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/809k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/756k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/289M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Model loaded : Helsinki-NLP/opus-mt-en-vi
Vocab size   : 53685
Model params : 127,122,944


In [6]:
MAX_LENGTH = 128

def preprocess(example):
    model_inputs = tokenizer(
        example["en"],
        max_length=MAX_LENGTH,
        truncation=True
    )
    labels = tokenizer(
        text_target=example["vi"],
        max_length=MAX_LENGTH,
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [7]:
tokenized_train = train_dataset.map(
    preprocess,
    batched=True,
    remove_columns=train_dataset.column_names
)
tokenized_valid = valid_dataset.map(
    preprocess,
    batched=True,
    remove_columns=valid_dataset.column_names
)
print("✅ Tokenization hoàn tất!")

Map:   0%|          | 0/140000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1268 [00:00<?, ? examples/s]

✅ Tokenization hoàn tất!


In [8]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

In [9]:
bleu_metric   = evaluate.load("sacrebleu")
meteor_metric = evaluate.load("meteor")
ter_metric    = evaluate.load("ter")
chrf_metric   = evaluate.load("chrf")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds       = [p.strip() for p in decoded_preds]
    decoded_labels_str  = [l.strip() for l in decoded_labels]
    decoded_labels_bleu = [[l] for l in decoded_labels_str]

    # --- BLEU ---
    bleu_result = bleu_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels_bleu
    )

    # --- METEOR ---
    meteor_result = meteor_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels_str
    )

    # --- TER ---
    ter_result = ter_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels_bleu
    )

    # --- chrF ---
    chrf_result = chrf_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels_bleu
    )

    return {
        "bleu":   round(bleu_result["score"], 4),        # 0-100, cao hơn tốt hơn
        "meteor": round(meteor_result["meteor"] * 100, 4), # 0-100, cao hơn tốt hơn
        "ter":    round(ter_result["score"], 4),          # 0-100, THẤP hơn tốt hơn
        "chrf":   round(chrf_result["score"], 4),         # 0-100, cao hơn tốt hơn
    }

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [10]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./envi-model",

    # --- Evaluation & Saving ---
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,

    # --- Batch size ---
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    # --- Optimizer ---
    learning_rate=2e-5,
    warmup_ratio=0.1,

    # --- Epochs ---
    num_train_epochs=5,

    # --- Generation ---
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=5,

    # --- Logging & Misc ---
    logging_steps=200,
    save_total_limit=2,
    fp16=True,
    report_to="none"
)

print("✅ Training args OK!")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✅ Training args OK!


In [11]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Bleu,Meteor,Ter,Chrf
1,3.138270,1.509064,33.986300,62.935600,53.016200,52.411300
2,2.924714,1.516720,33.994600,62.577800,52.911400,52.318800
3,2.764474,1.521488,33.570700,62.911100,52.789000,52.425200
4,2.627175,1.524795,33.580800,62.798300,52.869400,52.312700
5,2.562279,1.529556,34.061300,62.864300,52.495500,52.439400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_positions.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


TrainOutput(global_step=43750, training_loss=2.8230520047433036, metrics={'train_runtime': 9895.1429, 'train_samples_per_second': 70.742, 'train_steps_per_second': 4.421, 'total_flos': 1.0082174578458624e+16, 'train_loss': 2.8230520047433036, 'epoch': 5.0})

In [12]:
from bert_score import score as bert_score_fn

BATCH_SIZE  = 32
NUM_SAMPLES = 1268

raw_texts = [valid_dataset[i]["en"] for i in range(NUM_SAMPLES)]
raw_refs  = [valid_dataset[i]["vi"] for i in range(NUM_SAMPLES)]

# ── Sinh bản dịch ──────────────────────────────────────────────
all_preds = []
model.eval()

for i in range(0, NUM_SAMPLES, BATCH_SIZE):
    batch  = raw_texts[i : i + BATCH_SIZE]
    inputs = tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=128,
            num_beams=5,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    all_preds.extend(decoded)

    if (i // BATCH_SIZE) % 5 == 0:
        print(f"Đã xử lý {min(i + BATCH_SIZE, NUM_SAMPLES)}/{NUM_SAMPLES} câu...")

# ── BLEU ───────────────────────────────────────────────────────
bleu_result = bleu_metric.compute(
    predictions=all_preds,
    references=[[r] for r in raw_refs]
)

# ── METEOR ─────────────────────────────────────────────────────
meteor_result = meteor_metric.compute(
    predictions=all_preds,
    references=raw_refs
)

# ── BERTScore ──────────────────────────────────────────────────
print("\n⏳ Đang tính BERTScore (vài phút)...")
P, R, F1 = bert_score_fn(
    all_preds,
    raw_refs,
    lang="vi",
    rescale_with_baseline=True,
    verbose=True
)

# ── Tổng hợp ───────────────────────────────────────────────────
print("\n" + "="*50)
print("📋 TỔNG HỢP KẾT QUẢ ĐÁNH GIÁ")
print("="*50)
print(f"  BLEU      : {bleu_result['score']:.4f}  (0-100, càng cao càng tốt)")
print(f"  METEOR    : {meteor_result['meteor']*100:.4f}  (0-100, càng cao càng tốt)")
print(f"  BERTScore : {F1.mean():.4f}  (0-1,   càng cao càng tốt)")
print("="*50)


Đã xử lý 32/1268 câu...
Đã xử lý 192/1268 câu...
Đã xử lý 352/1268 câu...
Đã xử lý 512/1268 câu...
Đã xử lý 672/1268 câu...
Đã xử lý 832/1268 câu...
Đã xử lý 992/1268 câu...
Đã xử lý 1152/1268 câu...

⏳ Đang tính BERTScore (vài phút)...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/40 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/20 [00:00<?, ?it/s]

done in 5.29 seconds, 239.87 sentences/sec

📋 TỔNG HỢP KẾT QUẢ ĐÁNH GIÁ
  BLEU      : 34.2371  (0-100, càng cao càng tốt)
  METEOR    : 62.5843  (0-100, càng cao càng tốt)
  BERTScore : 0.8796  (0-1,   càng cao càng tốt)
